# Processar Tensor Mobilidade - Google Drive

Objetivo: baixar/processar o `tensor_mobilidade.csv` no ambiente do Google Colab e gerar um CSV leve com antenas + tecnologia predominante 3G/4G/5G.

Este notebook **não exige** que você organize pastas manualmente no Google Drive. Ele baixa a base leve do GitHub e o arquivo grande pelo ID do Google Drive.

## 1. Preparar ambiente

Instala o `gdown`, usado para baixar arquivo grande do Google Drive dentro do Colab.

In [ ]:
!pip -q install gdown
from pathlib import Path
import pandas as pd
import gdown

WORK_DIR = Path("/content/bit_app_dataset")
WORK_DIR.mkdir(parents=True, exist_ok=True)

ANTENAS = WORK_DIR / "antenas_flp.csv"
MOBILIDADE = WORK_DIR / "tensor_mobilidade.csv"
OUT_DIR = Path("/content/drive/MyDrive/NoCountry_BiT_App/processed")


## 2. Montar Google Drive para salvar o resultado

O arquivo grande será processado no ambiente temporário do Colab. O resultado leve será salvo no seu Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Saída será salva em:", OUT_DIR)


## 3. Baixar arquivo leve de antenas do GitHub


In [ ]:
ANTENAS_URL = "https://raw.githubusercontent.com/wongola-bit/appbit-hackathon/main/dataset-visent/referencias/antenas_flp.csv"
if not ANTENAS.exists():
    !wget -q -O "{ANTENAS}" "{ANTENAS_URL}"

print("ANTENAS existe:", ANTENAS.exists(), ANTENAS)
antenas = pd.read_csv(ANTENAS, dtype={"ecgi": str})
antenas.head()


## 4. Baixar tensor_mobilidade.csv pelo Google Drive

Arquivo grande: cerca de 2,63 GB. Ele será baixado para o disco temporário do Colab, não para seu computador.

In [ ]:
FILE_ID = "186vEhiwIgHOdj496Tv3nnEnQb_vumLXq"
if not MOBILIDADE.exists():
    gdown.download(id=FILE_ID, output=str(MOBILIDADE), quiet=False, fuzzy=True)

print("MOBILIDADE existe:", MOBILIDADE.exists(), MOBILIDADE)
print("Tamanho GB:", round(MOBILIDADE.stat().st_size / 1024**3, 2) if MOBILIDADE.exists() else None)


## 5. Processar em chunks

Lê apenas as colunas necessárias: `ecgi`, `rat_type` e `n_sessoes`.

In [ ]:
rat_map = {"WCDMA": "3G", "LTE": "4G", "NR": "5G"}
chunks = []
linhas = 0
usecols = ["ecgi", "rat_type", "n_sessoes"]

for chunk in pd.read_csv(MOBILIDADE, usecols=usecols, dtype={"ecgi": str, "rat_type": str}, chunksize=500_000):
    linhas += len(chunk)
    chunk["tecnologia"] = chunk["rat_type"].map(rat_map).fillna("OUTROS")
    agg = chunk.groupby(["ecgi", "tecnologia"], as_index=False)["n_sessoes"].sum()
    chunks.append(agg)
    print("linhas processadas:", linhas)

mobilidade_agg = pd.concat(chunks, ignore_index=True)
mobilidade_agg = mobilidade_agg.groupby(["ecgi", "tecnologia"], as_index=False)["n_sessoes"].sum()
mobilidade_agg.head()


## 6. Juntar tecnologias com coordenadas das antenas


In [ ]:
pivot = mobilidade_agg.pivot_table(index="ecgi", columns="tecnologia", values="n_sessoes", aggfunc="sum", fill_value=0).reset_index()
for col in ["3G", "4G", "5G", "OUTROS"]:
    if col not in pivot.columns:
        pivot[col] = 0

resultado = antenas.merge(pivot, on="ecgi", how="left")
for col in ["3G", "4G", "5G", "OUTROS"]:
    resultado[col] = resultado[col].fillna(0).astype(int)

resultado["total_sessoes"] = resultado[["3G", "4G", "5G", "OUTROS"]].sum(axis=1)
resultado["tecnologia_predominante"] = resultado[["3G", "4G", "5G", "OUTROS"]].idxmax(axis=1)
resultado.loc[resultado["total_sessoes"] == 0, "tecnologia_predominante"] = "SEM_DADO"
resultado = resultado.rename(columns={"3G": "sessoes_3g", "4G": "sessoes_4g", "5G": "sessoes_5g", "OUTROS": "sessoes_outros"})
resultado.head()


## 7. Salvar resultado leve no Drive


In [ ]:
out_file = OUT_DIR / "antenas_sinal_tratadas.csv"
resultado.to_csv(out_file, index=False, encoding="utf-8")
print("Arquivo gerado:", out_file)
print("Linhas:", len(resultado))
resultado[["ecgi", "cluster", "municipio", "lat", "lon", "sessoes_3g", "sessoes_4g", "sessoes_5g", "tecnologia_predominante"]].head(20)
